In [16]:
from datasets import load_dataset

vsr_dataset = load_dataset("cambridgeltl/vsr_zeroshot")

In [17]:
vsr_dataset['train'][0]

{'image': '000000558388.jpg',
 'image_link': 'http://images.cocodataset.org/train2017/000000558388.jpg',
 'caption': 'The cake is next to the person.',
 'label': 1,
 'relation': 'next to',
 'subj': 'cake',
 'obj': 'person',
 'annotator_id': 35,
 'vote_true_validator_id': '[2, 67, 20]',
 'vote_false_validator_id': '[]'}

In [ ]:
import os
import requests
from datasets import load_dataset, Dataset
from tqdm import tqdm

# Output directories
output_dir = "/home/saydalie/project/VLM-R1/data/images/vsr" # change this to your path where you want to save the images
images_dir = os.path.join(output_dir, "images")
os.makedirs(images_dir, exist_ok=True)

In [19]:
# download image and return local path
def download_and_replace_image(example):
    image_url = example["image_link"]
    image_name = example["image"]
    local_path = os.path.join(images_dir, image_name)
    
    # Download only if not already present
    if not os.path.exists(local_path):
        try:
            response = requests.get(image_url, timeout=10)
            response.raise_for_status()
            with open(local_path, "wb") as f:
                f.write(response.content)
        except Exception as e:
            print(f"Failed to download {image_url}: {e}")
            local_path = None  # or set to empty string if needed

    # Replace the 'image' field with the local file path
    return {
        "image_path": local_path,
        "caption": example["caption"],
        "label": example["label"],
        "relation": example["relation"],
        "subj": example["subj"],
        "obj": example["obj"]
    }

In [20]:
# Map the dataset with image downloading
vsr_dataset = vsr_dataset.map(download_and_replace_image)

Map: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1222/1222 [06:44<00:00,  3.02 examples/s]


In [33]:
# Remove unused columns (optional, in case you want a clean dataset)
vsr_dataset = vsr_dataset.remove_columns([col for col in vsr_dataset.column_names['train'] if col not in ["image_path", "caption", "label", "relation", "subj", "obj"]])

In [46]:
# Sample
vsr_dataset['train'][0]

{'image_path': '/home/saydalie/project/VLM-R1/data/images/vsr/000000558388.jpg',
 'caption': 'The cake is next to the person.',
 'label': 1,
 'relation': 'next to',
 'subj': 'cake',
 'obj': 'person'}

In [ ]:
# Save the dataset to disk
vsr_dataset.save_to_disk('/home/saydalie/project/VLM-R1/data/vsr') # # change this to your path where you want to save the dataset

Saving the dataset (1/1 shards): 100%|████████████████████████████████████████████████████████████████████████████| 1222/1222 [00:00<00:00, 207163.80 examples/s]
